In [8]:
# =========================
# 1. Imports and extraction
# =========================
from pathlib import Path
from itertools import chain
from collections import Counter
import tarfile
import os
import numpy as np

path_tofile = "./ebm_nlp_2_00.tar.gz"
extract_directory = os.path.dirname(path_tofile)

DATA_DIR = Path("./ebm_nlp_2_00")
docs_dir = DATA_DIR / "documents"

# 只有在数据目录不存在时才解压
if not DATA_DIR.exists():
    print("Extracting dataset...")
    if tarfile.is_tarfile(path_tofile):
        with tarfile.open(path_tofile) as f:
            f.extractall(path=extract_directory)
else:
    print("Dataset already extracted, skipping extraction.")

print("DATA_DIR exists:", DATA_DIR.exists())
print("documents dir exists:", docs_dir.exists())

Dataset already extracted, skipping extraction.
DATA_DIR exists: True
documents dir exists: True


In [9]:
# =========================
# 2. Helper functions
# =========================
def get_doc_ids(split="train", label_type="participants"):
    """
    split: 'train' or 'test'
    label_type: 'participants', 'interventions', 'outcomes'
    """
    if split == "test":
        split = "test/gold"

    ann_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
    )

    doc_ids = [p.stem.split(".")[0] for p in ann_dir.glob("*.AGGREGATED.ann")]
    return sorted(doc_ids)


def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]


def load_documents(doc_ids):
    return [load_document(doc_id) for doc_id in doc_ids]


def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    """
    Reads one hierarchical label file.
    """
    if split == "test":
        split = "test/gold"

    ann_path = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
        / f"{doc_id}.AGGREGATED.ann"
    )

    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None

    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]

    return labels


def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels


def hierarchical_to_bio(tags):
    """
    Convert EBM-NLP hierarchical labels (0-4) to BIO tags.
    0 -> O
    non-zero span start -> B
    non-zero span continuation -> I
    """
    bio = []
    prev = 0

    for t in tags:
        t = int(t)
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")
        prev = t

    return bio


def convert_all_labels_to_bio(labels):
    return [hierarchical_to_bio(doc_labels) for doc_labels in labels]


def merge_token_labels(p_labels, i_labels, o_labels):
    """
    Merge three BIO label streams into one final tag sequence.

    Priority:
    P > I > O > O

    Final labels:
    B-P, I-P, B-I, I-I, B-O, I-O, O
    """
    merged = []

    for p, i, o in zip(p_labels, i_labels, o_labels):
        if p == "B":
            merged.append("B-P")
        elif p == "I":
            merged.append("I-P")
        elif i == "B":
            merged.append("B-I")
        elif i == "I":
            merged.append("I-I")
        elif o == "B":
            merged.append("B-O")
        elif o == "I":
            merged.append("I-O")
        else:
            merged.append("O")

    return merged

In [10]:
# =========================
# 3. Load participants data
# =========================
doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)

participants_labels = load_labels(doc_ids_p, "participants", split="train")
test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")

participants_labels = convert_all_labels_to_bio(participants_labels)
test_participants_labels = convert_all_labels_to_bio(test_participants_labels)

print("Participants train docs:", len(doc_ids_p))
print("Participants test docs:", len(test_doc_ids_p))
print("Participants example tokens:", participants_tokens[0][:15])
print("Participants example labels:", participants_labels[0][:15])

Participants train docs: 4609
Participants test docs: 189
Participants example tokens: ['[', 'Triple', 'therapy', 'regimens', 'involving', 'H2', 'blockaders', 'for', 'therapy', 'of', 'Helicobacter', 'pylori', 'infections', ']', '.']
Participants example labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [11]:
# =========================
# 4. Load interventions data
# =========================
doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

interventions_labels = load_labels(doc_ids_i, "interventions", split="train")
test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")

interventions_labels = convert_all_labels_to_bio(interventions_labels)
test_interventions_labels = convert_all_labels_to_bio(test_interventions_labels)

print("Interventions train docs:", len(doc_ids_i))
print("Interventions test docs:", len(test_doc_ids_i))
print("Interventions example labels:", interventions_labels[0][:15])

Interventions train docs: 4746
Interventions test docs: 187
Interventions example labels: ['O', 'O', 'O', 'O', 'O', 'B', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [12]:
# =========================
# 5. Load outcomes data
# =========================
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")

outcomes_labels = convert_all_labels_to_bio(outcomes_labels)
test_outcomes_labels = convert_all_labels_to_bio(test_outcomes_labels)

print("Outcomes train docs:", len(doc_ids_o))
print("Outcomes test docs:", len(test_doc_ids_o))
print("Outcomes example labels:", outcomes_labels[0][:15])

Outcomes train docs: 4681
Outcomes test docs: 190
Outcomes example labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B', 'I', 'I', 'I', 'O']


In [13]:
# =========================
# 6. Keep shared documents only
# =========================
common_doc_ids = sorted(set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o))
print("Common train documents across P/I/O:", len(common_doc_ids))

# Build dictionaries for alignment
p_tokens_dict = dict(zip(doc_ids_p, participants_tokens))
p_labels_dict = dict(zip(doc_ids_p, participants_labels))

i_labels_dict = dict(zip(doc_ids_i, interventions_labels))
o_labels_dict = dict(zip(doc_ids_o, outcomes_labels))

Common train documents across P/I/O: 4457


In [14]:
# =========================
# 7. Build final token-level dataset
# =========================
all_token_docs = []
all_token_labels = []

for doc_id in common_doc_ids:
    tokens = p_tokens_dict[doc_id]  # tokens are the same document text
    p_labs = p_labels_dict[doc_id]
    i_labs = i_labels_dict[doc_id]
    o_labs = o_labels_dict[doc_id]

    merged_labels = merge_token_labels(p_labs, i_labs, o_labs)

    # safety check
    assert len(tokens) == len(merged_labels), f"Length mismatch in doc {doc_id}"

    all_token_docs.append(tokens)
    all_token_labels.append(merged_labels)

print("Final number of documents:", len(all_token_docs))
print("Example tokens:", all_token_docs[0][:20])
print("Example merged labels:", all_token_labels[0][:20])

Final number of documents: 4457
Example tokens: ['[', 'Triple', 'therapy', 'regimens', 'involving', 'H2', 'blockaders', 'for', 'therapy', 'of', 'Helicobacter', 'pylori', 'infections', ']', '.', 'Comparison', 'of', 'ranitidine', 'and', 'lansoprazole']
Example merged labels: ['O', 'O', 'O', 'O', 'O', 'B-I', 'I-I', 'O', 'O', 'O', 'B-O', 'I-O', 'I-O', 'I-O', 'O', 'B-O', 'O', 'B-I', 'O', 'B-I']


In [15]:
# =========================
# 8. Inspect label distribution
# =========================
label_counter = Counter(label for doc in all_token_labels for label in doc)
print("Merged label distribution:")
print(label_counter)

Merged label distribution:
Counter({'O': 973681, 'I-O': 79987, 'I-I': 53835, 'I-P': 35018, 'B-O': 30658, 'B-I': 29900, 'B-P': 15697})


In [16]:
# =========================
# 9. Build vocabularies
# =========================
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_LABEL = "<PAD>"

token2id = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for doc in all_token_docs:
    for tok in doc:
        if tok not in token2id:
            token2id[tok] = len(token2id)

id2token = {v: k for k, v in token2id.items()}

label_list = [PAD_LABEL, "O", "B-P", "I-P", "B-I", "I-I", "B-O", "I-O"]
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}

print("Vocab size:", len(token2id))
print("Label mapping:", label2id)

Vocab size: 49938
Label mapping: {'<PAD>': 0, 'O': 1, 'B-P': 2, 'I-P': 3, 'B-I': 4, 'I-I': 5, 'B-O': 6, 'I-O': 7}


In [17]:
# =========================
# 10. Encode tokens and labels
# =========================
def encode_tokens(tokens, token2id):
    return [token2id.get(tok, token2id[UNK_TOKEN]) for tok in tokens]

def encode_labels(labels, label2id):
    return [label2id[lab] for lab in labels]

encoded_docs = [encode_tokens(doc, token2id) for doc in all_token_docs]
encoded_labels = [encode_labels(labs, label2id) for labs in all_token_labels]

print("Encoded tokens example:", encoded_docs[0][:15])
print("Encoded labels example:", encoded_labels[0][:15])

Encoded tokens example: [2, 3, 4, 5, 6, 7, 8, 9, 4, 10, 11, 12, 13, 14, 15]
Encoded labels example: [1, 1, 1, 1, 1, 4, 5, 1, 1, 1, 6, 7, 7, 7, 1]


In [18]:
#test
print("Final number of documents:", len(all_token_docs))
print("Example tokens:", all_token_docs[0][:20])
print("Example merged labels:", all_token_labels[0][:20])
print(label_counter)
print("Vocab size:", len(token2id))
print("Label mapping:", label2id)

Final number of documents: 4457
Example tokens: ['[', 'Triple', 'therapy', 'regimens', 'involving', 'H2', 'blockaders', 'for', 'therapy', 'of', 'Helicobacter', 'pylori', 'infections', ']', '.', 'Comparison', 'of', 'ranitidine', 'and', 'lansoprazole']
Example merged labels: ['O', 'O', 'O', 'O', 'O', 'B-I', 'I-I', 'O', 'O', 'O', 'B-O', 'I-O', 'I-O', 'I-O', 'O', 'B-O', 'O', 'B-I', 'O', 'B-I']
Counter({'O': 973681, 'I-O': 79987, 'I-I': 53835, 'I-P': 35018, 'B-O': 30658, 'B-I': 29900, 'B-P': 15697})
Vocab size: 49938
Label mapping: {'<PAD>': 0, 'O': 1, 'B-P': 2, 'I-P': 3, 'B-I': 4, 'I-I': 5, 'B-O': 6, 'I-O': 7}


train / dev / test split

In [19]:
from sklearn.model_selection import train_test_split

indices = list(range(len(encoded_docs)))

train_idx, test_idx = train_test_split(
    indices, test_size=0.15, random_state=42
)
train_idx, dev_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_docs = [encoded_docs[i] for i in train_idx]
train_labels = [encoded_labels[i] for i in train_idx]

dev_docs = [encoded_docs[i] for i in dev_idx]
dev_labels = [encoded_labels[i] for i in dev_idx]

test_docs = [encoded_docs[i] for i in test_idx]
test_labels = [encoded_labels[i] for i in test_idx]

print("Train docs:", len(train_docs))
print("Dev docs:", len(dev_docs))
print("Test docs:", len(test_docs))

Train docs: 3219
Dev docs: 569
Test docs: 669


padding and mask

In [24]:
PAD_TOKEN = "<PAD>"
PAD_LABEL = "<PAD>"

MAX_LEN = 256    # adjustable

def pad_sequence(seq, max_len, pad_value):
    if len(seq) < max_len:
        return seq + [pad_value] * (max_len - len(seq))
    return seq[:max_len]

def make_attention_mask(seq, max_len):
    if len(seq) < max_len:
        return [1] * len(seq) + [0] * (max_len - len(seq))
    return [1] * max_len

DataLoader

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader

class PICODataset(Dataset):
    def __init__(self, docs, labels, max_len=256):
        self.docs = docs
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        token_ids = self.docs[idx]
        label_ids = self.labels[idx]

        attn_mask = make_attention_mask(token_ids, self.max_len)

        token_ids = pad_sequence(token_ids, self.max_len, token2id[PAD_TOKEN])
        label_ids = pad_sequence(label_ids, self.max_len, label2id[PAD_LABEL])

        return {
            "input_ids": torch.tensor(token_ids, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long),
            "mask": torch.tensor(attn_mask, dtype=torch.bool),
        }

BATCH_SIZE = 16

train_dataset = PICODataset(train_docs, train_labels, max_len=MAX_LEN)
dev_dataset = PICODataset(dev_docs, dev_labels, max_len=MAX_LEN)
test_dataset = PICODataset(test_docs, test_labels, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [23]:
batch = next(iter(train_loader))

print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)
print("mask shape:", batch["mask"].shape)

print("Example input_ids:", batch["input_ids"][0][:20])
print("Example labels:", batch["labels"][0][:20])
print("Example mask:", batch["mask"][0][:20])

input_ids shape: torch.Size([16, 256])
labels shape: torch.Size([16, 256])
mask shape: torch.Size([16, 256])
Example input_ids: tensor([2506,   10, 1391, 8523,  645,  229,  329,  330,   20, 1985,  331, 2531,
          15, 3579, 1985,   36, 3789,  331, 2531, 1779])
Example labels: tensor([6, 1, 1, 1, 1, 1, 6, 7, 1, 1, 2, 3, 1, 2, 1, 1, 1, 1, 1, 1])
Example mask: tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True])


In [ ]:
import gensim.downloader as api
import numpy as np

USE_PRETRAINED = False  #  True if used

embedding_matrix = None
EMBED_DIM = 100  #.eg glove-wiki-gigaword-100

if USE_PRETRAINED:
    print("Loading pretrained embeddings...")
    wv = api.load("glove-wiki-gigaword-100")

    embedding_matrix = np.random.normal(scale=0.6, size=(len(token2id), EMBED_DIM)).astype(np.float32)
    embedding_matrix[token2id[PAD_TOKEN]] = np.zeros(EMBED_DIM, dtype=np.float32)

    found = 0
    for tok, idx in token2id.items():
        if tok in wv:
            embedding_matrix[idx] = wv[tok]
            found += 1

    print(f"Matched pretrained vectors: {found}/{len(token2id)}")

BiLSTM token tagger

In [25]:
import torch.nn as nn

class BiLSTMTagger(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim=100,
        hidden_dim=128,
        pretrained_matrix=None,
        freeze_embeddings=False,
        dropout=0.2,
        pad_idx=0
    ):
        super().__init__()

        if pretrained_matrix is not None:
            emb_tensor = torch.tensor(pretrained_matrix, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                emb_tensor,
                freeze=freeze_embeddings,
                padding_idx=pad_idx
            )
            embedding_dim = emb_tensor.shape[1]
        else:
            self.embedding = nn.Embedding(
                vocab_size,
                embedding_dim,
                padding_idx=pad_idx
            )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, mask=None):
        x = self.embedding(input_ids)     # (batch, seq, emb_dim)
        x, _ = self.lstm(x)               # (batch, seq, 2*hidden_dim)
        x = self.dropout(x)
        logits = self.classifier(x)       # (batch, seq, num_labels)
        return logits

In [ ]:
define weighted loss & focal loss

In [26]:
import torch
import numpy as np

def compute_class_weights(train_labels, num_labels, pad_label_id):
    counts = np.zeros(num_labels, dtype=np.float32)

    for seq in train_labels:
        for lab in seq:
            if lab != pad_label_id:
                counts[lab] += 1

    counts[counts == 0] = 1.0
    weights = counts.sum() / counts
    weights = weights / weights.mean()
    weights[pad_label_id] = 0.0
    return torch.tensor(weights, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, ignore_index=0):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.ce = nn.CrossEntropyLoss(reduction="none", ignore_index=ignore_index)

    def forward(self, logits, targets):
        # logits: (batch, seq, num_labels)
        # targets: (batch, seq)
        batch, seq, num_labels = logits.shape
        logits = logits.view(-1, num_labels)
        targets = targets.view(-1)

        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        valid = targets != self.ignore_index
        if valid.sum() == 0:
            return focal_loss.mean()
        return focal_loss[valid].mean()

In [32]:
import gensim.downloader as api
import numpy as np

print("Loading pretrained embeddings...")
wv = api.load("glove-wiki-gigaword-100")

EMBED_DIM = 100
embedding_matrix = np.random.normal(
    scale=0.6,
    size=(len(token2id), EMBED_DIM)
).astype(np.float32)

embedding_matrix[token2id["<PAD>"]] = np.zeros(EMBED_DIM, dtype=np.float32)

found = 0
for tok, idx in token2id.items():
    if tok in wv:
        embedding_matrix[idx] = wv[tok]
        found += 1

print(f"Matched pretrained vectors: {found}/{len(token2id)}")
print("Embedding matrix shape:", embedding_matrix.shape)

Loading pretrained embeddings...
[=========-----------------------------------------] 18.3% 23.4/128.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[========================--------------------------] 48.1% 61.6/128.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=======================================-----------] 78.9% 101.0/128.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 128.1/128.1MB downloaded
Matched pretrained vectors: 20023/49938
Embedding matrix shape: (49938, 100)


In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

USE_PRETRAINED = True   # True for pretrained
LOSS_TYPE = "focal"   # "weighted" or "focal"

EMBED_DIM = 100
HIDDEN_DIM = 256
PAD_LABEL_ID = label2id["<PAD>"]

class_weights = compute_class_weights(
    train_labels,
    num_labels=len(label2id),
    pad_label_id=PAD_LABEL_ID
).to(device)

print("Class weights:", class_weights)

model = BiLSTMTagger(
    vocab_size=len(token2id),
    num_labels=len(label2id),
    embedding_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    pretrained_matrix=embedding_matrix if USE_PRETRAINED else None,
    freeze_embeddings=False,
    dropout=0.3,
    pad_idx=token2id["<PAD>"]
).to(device)

if LOSS_TYPE == "weighted":
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        ignore_index=PAD_LABEL_ID
    )
elif LOSS_TYPE == "focal":
    criterion = FocalLoss(ignore_index=PAD_LABEL_ID)
else:
    raise ValueError("LOSS_TYPE must be 'weighted' or 'focal'")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Device: cuda
Class weights: tensor([0.0000e+00, 1.1334e-05, 7.0553e-04, 3.1294e-04, 3.7063e-04, 2.0580e-04,
        3.6185e-04, 1.3871e-04], device='cuda:0')


define train and evaluate

In [52]:
from sklearn.metrics import f1_score, classification_report

def compute_loss(logits, labels, criterion):
    if isinstance(criterion, nn.CrossEntropyLoss):
        batch, seq, num_labels = logits.shape
        return criterion(logits.view(-1, num_labels), labels.view(-1))
    else:
        return criterion(logits, labels)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = compute_loss(logits, labels, criterion)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_gold = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)

            logits = model(input_ids)
            loss = compute_loss(logits, labels, criterion)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)

            valid = mask.view(-1)
            preds_flat = preds.view(-1)[valid]
            labels_flat = labels.view(-1)[valid]

            keep = labels_flat != label2id["<PAD>"]
            preds_flat = preds_flat[keep]
            labels_flat = labels_flat[keep]

            all_preds.extend(preds_flat.cpu().tolist())
            all_gold.extend(labels_flat.cpu().tolist())

    macro_f1 = f1_score(all_gold, all_preds, average="macro")
    return total_loss / len(dataloader), macro_f1, all_gold, all_preds

epoch 5 training

In [68]:
NUM_EPOCHS = 5
best_dev_f1 = -1
best_state = None

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    dev_loss, dev_f1, _, _ = evaluate(model, dev_loader, criterion, device)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train loss: {train_loss:.4f}")
    print(f"  Dev loss:   {dev_loss:.4f}")
    print(f"  Dev F1:     {dev_f1:.4f}")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print("Best dev F1:", best_dev_f1)

Epoch 1/5
  Train loss: 0.4147
  Dev loss:   0.3058
  Dev F1:     0.3347
Epoch 2/5
  Train loss: 0.2606
  Dev loss:   0.2555
  Dev F1:     0.4519
Epoch 3/5
  Train loss: 0.2235
  Dev loss:   0.2496
  Dev F1:     0.5106
Epoch 4/5
  Train loss: 0.1968
  Dev loss:   0.2388
  Dev F1:     0.5083
Epoch 5/5
  Train loss: 0.1774
  Dev loss:   0.2376
  Dev F1:     0.5359
Best dev F1: 0.5359103740811849


evaluate

In [69]:
model.load_state_dict(best_state)

test_loss, test_f1, gold, preds = evaluate(model, test_loader, criterion, device)

print("Test loss:", test_loss)
print("Test macro F1:", test_f1)

target_names = [lab for lab in label_list if lab != "<PAD>"]
target_ids = [label2id[lab] for lab in target_names]

print(classification_report(
    gold, preds,
    labels=target_ids,
    target_names=target_names,
    digits=4
))

Test loss: 0.22044035224687486
Test macro F1: 0.5446348617205756
              precision    recall  f1-score   support

           O     0.8751    0.9177    0.8959    115670
         B-P     0.4866    0.4923    0.4895      2220
         I-P     0.5717    0.3018    0.3951      4877
         B-I     0.5408    0.5587    0.5496      4140
         I-I     0.5109    0.4202    0.4611      7452
         B-O     0.4851    0.4691    0.4770      3688
         I-O     0.6145    0.4886    0.5443     10044

    accuracy                         0.8157    148091
   macro avg     0.5835    0.5212    0.5446    148091
weighted avg     0.8042    0.8157    0.8075    148091



epoch 3 pretrained, focal

Test loss: 0.2327716066723778
Test macro F1: 0.49721087080046
              precision    recall  f1-score   support

           O     0.8511    0.9418    0.8942    115670
         B-P     0.5483    0.3554    0.4313      2220
         I-P     0.5453    0.2889    0.3777      4877
         B-I     0.5932    0.4343    0.5015      4140
         I-I     0.5458    0.3025    0.3892      7452
         B-O     0.5330    0.2977    0.3820      3688
         I-O     0.6222    0.4244    0.5046     10044

    accuracy                         0.8140    148091
   macro avg     0.6055    0.4350    0.4972    148091
weighted avg     0.7905    0.8140    0.7947    148091

The BiLSTM model with pretrained embeddings and focal loss achieves a macro F1 score of 0.497 on the test set, significantly outperforming the weighted loss baseline (0.372). 

Compared to the weighted loss setup, which tends to over-predict entity labels and results in high recall but low precision, the focal loss introduces a better balance between precision and recall. In particular, precision for entity classes increases substantially (e.g., B-P from ~0.18 to 0.55), while recall decreases to a more reasonable level, leading to improved overall F1 scores.

Furthermore, the use of pretrained GloVe embeddings enhances the model's ability to capture semantic relationships between tokens, which is especially beneficial in domain-specific text such as biomedical abstracts. 

Notably, the performance on the majority class "O" improves dramatically, with recall increasing from approximately 0.43 to 0.94. This indicates that the model no longer suffers from excessive bias toward minority classes, which was observed in the weighted loss setup.

Overall, the combination of pretrained embeddings and focal loss results in a more balanced and robust model for token-level classification.

pretrained, focal best data yet 
NUM_EPOCHS = 5
HIDDEN_DIM = 256
lr = 5e-4
dropout = 0.3


Test loss: 0.22044035224687486
Test macro F1: 0.5446348617205756
              precision    recall  f1-score   support

           O     0.8751    0.9177    0.8959    115670
         B-P     0.4866    0.4923    0.4895      2220
         I-P     0.5717    0.3018    0.3951      4877
         B-I     0.5408    0.5587    0.5496      4140
         I-I     0.5109    0.4202    0.4611      7452
         B-O     0.4851    0.4691    0.4770      3688
         I-O     0.6145    0.4886    0.5443     10044

    accuracy                         0.8157    148091
   macro avg     0.5835    0.5212    0.5446    148091
weighted avg     0.8042    0.8157    0.8075    148091


Evaluation

token to span

In [70]:
def bio_to_spans(label_ids, id2label):
    """
    input:
        label_ids: token id，like [1, 1, 2, 3, 1, 4, ...]
       

    output:
        spans: [('P', start, end), ('I', start, end), ('O', start, end)]
    """
    labels = [id2label[i] for i in label_ids]
    spans = []
    i = 0

    while i < len(labels):
        lab = labels[i]

        if lab == "O" or lab == "<PAD>":
            i += 1
            continue

        if lab.startswith("B-"):
            ent_type = lab.split("-")[1]
            start = i
            end = i
            i += 1

            while i < len(labels) and labels[i] == f"I-{ent_type}":
                end = i
                i += 1

            spans.append((ent_type, start, end))

        elif lab.startswith("I-"):
            # 容错：如果不小心遇到孤立的 I-X，也把它当成一个新span
            ent_type = lab.split("-")[1]
            start = i
            end = i
            i += 1

            while i < len(labels) and labels[i] == f"I-{ent_type}":
                end = i
                i += 1

            spans.append((ent_type, start, end))

        else:
            i += 1

    return spans

In [71]:
def predict_document_label_ids(model, token_ids, max_len=256):
    """
    对单篇文档做预测，返回预测标签id序列（去掉padding部分）
    """
    model.eval()

    mask = make_attention_mask(token_ids, max_len)
    padded = pad_sequence(token_ids, max_len, token2id["<PAD>"])

    x = torch.tensor([padded], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x)
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()

    valid_len = sum(mask)
    preds = preds[:valid_len]
    return preds

In [72]:
def get_doc_gold_pred_spans(model, token_ids, gold_label_ids, id2label, max_len=256):
    """
    输入:
        token_ids: 一篇文档的token id列表
        gold_label_ids: 这篇文档的gold标签id列表

    输出:
        gold_spans, pred_spans
    """
    pred_label_ids = predict_document_label_ids(model, token_ids, max_len=max_len)

    gold_spans = bio_to_spans(gold_label_ids, id2label)
    pred_spans = bio_to_spans(pred_label_ids, id2label)

    return gold_spans, pred_spans

In [73]:
all_gold_spans = []
all_pred_spans = []

for token_ids, gold_label_ids in zip(test_docs, test_labels):
    gold_spans, pred_spans = get_doc_gold_pred_spans(
        model=model,
        token_ids=token_ids,
        gold_label_ids=gold_label_ids,
        id2label=id2label,
        max_len=MAX_LEN
    )
    all_gold_spans.append(gold_spans)
    all_pred_spans.append(pred_spans)

print("Number of test documents:", len(all_gold_spans))
print("Example gold spans:", all_gold_spans[0][:10])
print("Example pred spans:", all_pred_spans[0][:10])

Number of test documents: 669
Example gold spans: [('P', 0, 1), ('P', 7, 11), ('P', 14, 15), ('P', 24, 27), ('P', 43, 44), ('O', 70, 71), ('I', 73, 77), ('O', 94, 95), ('P', 123, 125)]
Example pred spans: [('O', 24, 25), ('O', 70, 71), ('O', 94, 95)]


In [74]:
def spans_to_text(tokens, spans):
    """
    把 span 的 start/end 映射回实际文本
    输入:
        tokens: 一篇文档的token列表
        spans: [('P', 10, 12), ...]
    输出:
        [('P', 10, 12, 'actual text'), ...]
    """
    results = []
    for ent_type, start, end in spans:
        span_text = " ".join(tokens[start:end+1])
        results.append((ent_type, start, end, span_text))
    return results

In [75]:
example_idx = 0

example_tokens = all_token_docs[test_idx[example_idx]]
example_gold_spans = all_gold_spans[example_idx]
example_pred_spans = all_pred_spans[example_idx]

print("GOLD SPANS WITH TEXT:")
for item in spans_to_text(example_tokens, example_gold_spans)[:10]:
    print(item)

print("\nPRED SPANS WITH TEXT:")
for item in spans_to_text(example_tokens, example_pred_spans)[:10]:
    print(item)

GOLD SPANS WITH TEXT:
('P', 0, 1, 'Anxiety sensitivity')
('P', 7, 11, 'later anxiety symptoms and syndromes')
('P', 14, 15, 'anxiety sensitivity')
('P', 24, 27, 'anxiety symptoms and panic')
('P', 43, 44, 'anxiety psychopathology')
('O', 70, 71, 'anxiety symptoms')
('I', 73, 77, 'controlling for potential confounding factors')
('O', 94, 95, 'anxiety symptoms')
('P', 123, 125, 'anxiety symptoms .')

PRED SPANS WITH TEXT:
('O', 24, 25, 'anxiety symptoms')
('O', 70, 71, 'anxiety symptoms')
('O', 94, 95, 'anxiety symptoms')


Due to the limitations of the dataset, comparator entities are not explicitly annotated. Therefore, this work focuses on extracting Participants, Interventions, and Outcomes (PIO), where comparator information is implicitly included within intervention mentions.